# Predictive Student Performance Analysis System

This notebook demonstrates the Exploratory Data Analysis (EDA), feature analysis, and model training workflow for predicting student academic performance and risk assessment. We use synthetic data engineered to resemble standard academic patterns (attendance, study habits, failures, parental support, test prep, etc.).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Load and Clean Data

In [ ]:
# Load dataset
data_path = "../data/student_data.csv"
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Summary statistics
df.describe()

## 2. Exploratory Data Analysis (EDA)
We investigate key factors that influence a student's final grade and pass rate.

In [ ]:
# Distribution of Final Grades
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="FinalGrade", kde=True, bins=25, color="#4C72B0")
plt.axvline(60, color="red", linestyle="--", label="Passing Threshold (60)")
plt.title("Distribution of Final Grades")
plt.xlabel("Final Grade")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# Attendance vs Final Grade
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="AttendanceRate", y="FinalGrade", hue="Pass", palette={0: "red", 1: "green"}, alpha=0.6)
plt.title("Impact of Attendance on Final Grade")
plt.xlabel("Attendance Rate (%)")
plt.ylabel("Final Grade")
plt.show()

In [ ]:
# Study Time vs Final Grade
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="Failures", y="FinalGrade", palette="Oranges")
plt.title("Impact of Previous Academic Failures on Final Grade")
plt.xlabel("Number of Past Failures")
plt.ylabel("Final Grade")
plt.show()

## 3. Model Training & Evaluation

In [ ]:
# Select features and split dataset
feature_cols = [
    "StudyTimeWeekly", "AttendanceRate", "SleepHours", "Failures", 
    "ParentalSupport", "Extracurriculars", "Tutoring", "TestPrepCourse", "PreviousGrade"
]

X = df[feature_cols]
y_grade = df["FinalGrade"]
y_pass = df["Pass"]

X_train, X_test, y_train_grade, y_test_grade = train_test_split(X, y_grade, test_size=0.2, random_state=42)
_, _, y_train_pass, y_test_pass = train_test_split(X, y_pass, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Regressor for final grade prediction
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train_scaled, y_train_grade)
y_pred_grade = regressor.predict(X_test_scaled)

print(f"Regression Mean Squared Error: {mean_squared_error(y_test_grade, y_pred_grade):.2f}")
print(f"Regression R2 Score: {r2_score(y_test_grade, y_pred_grade):.2f}")

In [ ]:
# Train Classifier for risk (Pass/Fail) assessment
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train_scaled, y_train_pass)
y_pred_pass = classifier.predict(X_test_scaled)

print(f"Classification Accuracy: {accuracy_score(y_test_pass, y_pred_pass):.2f}")
print("\nClassification Report:")
print(classification_report(y_test_pass, y_pred_pass))

# Feature Importance Visualization
plt.figure(figsize=(10, 6))
importances = regressor.feature_importances_
indices = np.argsort(importances)[::-1]
sns.barplot(x=importances[indices], y=[feature_cols[i] for i in indices], palette="viridis")
plt.title("Feature Importance - Predictors of Final Grades")
plt.xlabel("Relative Importance")
plt.show()